In [1]:
import os
os.listdir()

['.config',
 'X_train.csv',
 'X_test.csv',
 'y_test.csv',
 'y_train.csv',
 'final_database_Comp333.csv',
 'sample_data']

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

# 1. LOAD RAW-INTEGRATED DATA
df_raw = pd.read_csv("final_database_Comp333.csv")

# 2. CREATE TARGET: delayed = 1 if ARR_DELAY >= 15, else 0
df_raw["delayed"] = (df_raw["ARR_DELAY"] >= 15).astype(int)

# 3. SELECT A SIMPLE SET OF FEATURES (before serious cleaning)
#    We avoid using TOTAL_DELAY as a feature because it is too close to the label.
features = [
    "CARRIER_CODE",  # airline
    "ORIGIN",        # origin airport
    "DEST",          # destination airport
    "CRS_DEP_TIME",  # scheduled departure time
]

X_raw = df_raw[features].copy()
y_raw = df_raw["delayed"]

# 4. ENCODE CATEGORICAL (minimal transformation so the model can run)
le_carrier = LabelEncoder()
le_origin = LabelEncoder()
le_dest = LabelEncoder()

X_raw["CARRIER_CODE"] = le_carrier.fit_transform(X_raw["CARRIER_CODE"])
X_raw["ORIGIN"]       = le_origin.fit_transform(X_raw["ORIGIN"])
X_raw["DEST"]         = le_dest.fit_transform(X_raw["DEST"])

# 5. TRAIN/TEST SPLIT
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# 6. TRAIN BASELINE MODEL
baseline_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
baseline_tree.fit(X_train_raw, y_train_raw)

# 7. EVALUATE
y_pred_raw = baseline_tree.predict(X_test_raw)
acc_raw = accuracy_score(y_test_raw, y_pred_raw)
f1_raw = f1_score(y_test_raw, y_pred_raw)

print("BEFORE cleaning/transformation:")
print("Accuracy:", acc_raw)
print("F1 score:", f1_raw)

BEFORE cleaning/transformation:
Accuracy: 0.7974999634924576
F1 score: 0.0


In [4]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. LOAD TRANSFORMED DATA
# Use na_values to convert '-' to NaN during loading
X_train = pd.read_csv("X_train.csv", na_values=['-'])
X_test  = pd.read_csv("X_test.csv", na_values=['-'])
y_train = pd.read_csv("y_train.csv")["delayed"]
y_test  = pd.read_csv("y_test.csv")["delayed"]

# Convert columns to numeric, coercing any remaining non-numeric to NaN,
# and then fill NaN values with the median of each column.
# This ensures all feature columns are numeric before model training.
for col in X_train.columns:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    if X_train[col].isnull().any():
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)

for col in X_test.columns:
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
    if X_test[col].isnull().any():
        median_val = X_test[col].median()
        X_test[col] = X_test[col].fillna(median_val)

# FIX: Align the lengths of X_train and y_train to prevent ValueError
min_len_train = min(len(X_train), len(y_train))
X_train = X_train.iloc[:min_len_train]
y_train = y_train.iloc[:min_len_train]

min_len_test = min(len(X_test), len(y_test))
X_test = X_test.iloc[:min_len_test]
y_test = y_test.iloc[:min_len_test]

# 2. TRAIN MODEL ON CLEANED + TRANSFORMED DATA
tree_clean = DecisionTreeClassifier(max_depth=6, random_state=42)
tree_clean.fit(X_train, y_train)

# 3. EVALUATE
y_pred_clean = tree_clean.predict(X_test)
acc_clean = accuracy_score(y_test, y_pred_clean)
f1_clean = f1_score(y_test, y_pred_clean)

print("AFTER cleaning/transformation:")
print("Accuracy:", acc_clean)
print("F1 score:", f1_clean)

AFTER cleaning/transformation:
Accuracy: 0.953606538941861
F1 score: 0.9091622847044167


In [5]:
def predict_delay_example(
    carrier_code, origin_code, dest_code,
    arr_time, arr_delay,
    carrier_delay, weather_delay, nas_delay,
    security_delay, late_aircraft_delay,
    day_of_week
):
    # One row with the same feature order as X_train
    row = [[
        carrier_code, origin_code, dest_code,
        arr_time, arr_delay,
        carrier_delay, weather_delay, nas_delay,
        security_delay, late_aircraft_delay,
        day_of_week
    ]]
    prediction = tree_clean.predict(row)[0]
    return "Delayed" if prediction == 1 else "On time"

# Example (you can change the numbers to something realistic)
print(predict_delay_example(
    carrier_code=1,
    origin_code=70,
    dest_code=201,
    arr_time=1800,
    arr_delay=11,
    carrier_delay=0,
    weather_delay=0,
    nas_delay=0,
    security_delay=0,
    late_aircraft_delay=5,
    day_of_week=6
))

Delayed


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


In [8]:
def predict_delay_example(
    carrier_code, origin_code, dest_code,
    arr_time, arr_delay,
    carrier_delay, weather_delay, nas_delay,
    security_delay, late_aircraft_delay,
    day_of_week
):
    # One row with the same feature order as X_train
    row = [[
        carrier_code, origin_code, dest_code,
        arr_time, arr_delay,
        carrier_delay, weather_delay, nas_delay,
        security_delay, late_aircraft_delay,
        day_of_week
    ]]
    prediction = tree_clean.predict(row)[0]
    return "Delayed" if prediction == 1 else "On time"

# Example (you can change the numbers to something realistic)
print(predict_delay_example(
    carrier_code=1,
    origin_code=70,
    dest_code=201,
    arr_time=1800,
    arr_delay=1,
    carrier_delay=10,
    weather_delay=3,
    nas_delay=0,
    security_delay=0,
    late_aircraft_delay=1,
    day_of_week=6
))

On time


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
